# Zero-Shot Cloud Classification with Qwen3.5-27B via OpenRouter (11 Standard Cloud Classes)

This notebook uses the **OpenRouter API** to call **`qwen/qwen3.5-27b`** for zero-shot cloud image classification using **11 standard cloud classes**: Altocumulus, Altostratus, Cirrocumulus, Cirrus, Clear Sky, Contrails, Cumulonimbus, Cumulus, Stratocumulus, Stratus, and Veil. Images are filtered from the local 15-class dataset to retain only those 11 classes. The model classifies them using the structured prompt with `<answer>` tags.

**No GPU required** — all inference runs remotely through OpenRouter. Designed to run locally (no Colab/Drive needed).

## 1. Install dependencies

In [ ]:
"""import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--break-system-packages",
    "numpy", "httpx", "datasets", "scikit-learn",
    "seaborn", "matplotlib", "pillow", "tqdm",
])
print("Dependencies installed.")
!pip3 install numpy"""

## 2. Configuration

In [ ]:
import os, re, random, math
import numpy as np

OPENROUTER_API_KEY = "---"
MODEL = "qwen/qwen3.5-27b"

DATASET_PATH = "/Users/nachomorerabarrios/Documentos/TFM/CloudDataset_Fixed"
OUTPUT_DIR = "/Users/nachomorerabarrios/Documentos/TFM/qwen35-27b-openrouter-zeroshot-11cloudclasses-classification"
TAG = "zeroshot_27b_11cloudclasses_classification"

# 11 standard cloud classes (subset of the 15-class dataset; 'Veil Clouds' renamed to 'Veil')
CLOUD_CLASSES = [
    "Altocumulus",
    "Altostratus",
    "Cirrocumulus",
    "Cirrus",
    "Clear Sky",
    "Contrails",
    "Cumulonimbus",
    "Cumulus",
    "Stratocumulus",
    "Stratus",
    "Veil",
]

# Use the full test set
NUM_TEST_IMAGES = None
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Model: {MODEL}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Cloud classes ({len(CLOUD_CLASSES)}): {CLOUD_CLASSES}")
print(f"Num test images: {'all' if NUM_TEST_IMAGES is None else NUM_TEST_IMAGES}")


## 3. Verify dataset path

In [ ]:
assert os.path.isdir(DATASET_PATH), f"Dataset not found at {DATASET_PATH}"
print(f"Dataset found: {DATASET_PATH}")
for entry in sorted(os.listdir(DATASET_PATH)):
    full = os.path.join(DATASET_PATH, entry)
    suffix = "/" if os.path.isdir(full) else ""
    print(f"  - {entry}{suffix}")

## 4. Load dataset & filter to 11 standard cloud classes

In [ ]:
from datasets import ClassLabel, load_dataset

np.random.seed(SEED)

print("Loading dataset...")
dataset = load_dataset(
    "imagefolder",
    data_files={
        "train": f"{DATASET_PATH}/train/**",
        "validation": f"{DATASET_PATH}/val/**",
        "test": f"{DATASET_PATH}/test/**",
    },
)

original_labels = dataset["train"].features["label"].names
print(f"Original classes ({len(original_labels)}): {original_labels}")

# Map 'Veil Clouds' → 'Veil'; all other classes in CLOUD_CLASSES keep their name.
# Classes not in CLOUD_CLASSES will be filtered out.
rename_map = {"Veil Clouds": "Veil"}

def remap_label(orig_name):
    return rename_map.get(orig_name, orig_name)

# Build label mapping: original label index → new label index (or None to discard)
cloud_classes_set = set(CLOUD_CLASSES)
orig_to_new = {}  # original label name → new label name (None = discard)
for lbl in original_labels:
    mapped = remap_label(lbl)
    orig_to_new[lbl] = mapped if mapped in cloud_classes_set else None

new_labels = CLOUD_CLASSES  # fixed ordered list
label2id = {l: i for i, l in enumerate(new_labels)}

def remap_example(ex):
    orig = original_labels[ex["label"]]
    new = orig_to_new[orig]
    ex["label"] = label2id[new] if new is not None else -1
    return ex

dataset = dataset.map(remap_example)

# Filter out discarded classes (label == -1)
dataset = dataset.filter(lambda ex: ex["label"] != -1)

new_features = dataset["train"].features.copy()
new_features["label"] = ClassLabel(names=new_labels)
dataset = dataset.cast(new_features)

NUM_LABELS = len(new_labels)
id2label = {i: l for i, l in enumerate(new_labels)}
VALID_CLASSES_LOWER = {l.lower(): l for l in new_labels}

print(f"\nCloud classes ({NUM_LABELS}): {new_labels}")
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")


## 5. Select test set

In [ ]:
from collections import Counter

test_split = dataset["test"]

if NUM_TEST_IMAGES is not None:
    indices = np.random.choice(len(test_split), size=NUM_TEST_IMAGES, replace=False)
    indices.sort()
    test_split = test_split.select(indices)

print(f"Test set: {len(test_split)} images")
label_counts = Counter(id2label[ex["label"]] for ex in test_split)
for label in new_labels:
    print(f"  {label}: {label_counts.get(label, 0)}")


## 6. Define classification prompt (single-label output)

In [ ]:
_labels_bullet = "\n".join(f"- {l}" for l in new_labels)

SYSTEM_PROMPT = (
    "You are a cloud classification expert with a lot of expertise on weather, physics, and clouds. "
    "Your goal is to, given a cloud image, classify it with one of the following classes:\n\n"
    + _labels_bullet + "\n\n"
    "Classify the cloud type in this image. You should justify your answer. "
    "The class must be included between the tags <answer></answer> with the strict same format as the labels I gave you. "
    "The tags with the answer must be placed at the end of your answer."
)
USER_PROMPT = "Classify the cloud type in this image."

print(f"System prompt:\n{SYSTEM_PROMPT}")
print(f"\nUser prompt: {USER_PROMPT}")
print("\nExpected output format:")
print("  This image shows ... <answer>Cumulus</answer>")


## 7. OpenRouter API client

In [ ]:
import base64
import io
import time
import json as _json
import httpx
from PIL import Image

# Shared httpx client — keeps TCP connections alive across requests
_http_client = httpx.Client(timeout=120, limits=httpx.Limits(max_connections=10))


def classify_image_openrouter(pil_image: Image.Image, max_retries: int = 6) -> str:
    img = pil_image.convert("RGB")
    img.thumbnail((1024, 1024), Image.LANCZOS)
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=85)
    b64 = base64.b64encode(buffer.getvalue()).decode()

    payload = {
        "model": MODEL,
        "thinking": {"type": "disabled"},  # disable Qwen3.5 thinking tokens so content is never empty
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                    {"type": "text", "text": USER_PROMPT},
                ],
            },
        ],
    }

    last_error = None
    for attempt in range(max_retries):
        try:
            response = _http_client.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
                json=payload,
            )
            data = response.json()
            if response.status_code == 429:
                retry_after = int(data.get("error", {}).get("metadata", {}).get("retry_after_seconds", 10))
                wait = max(retry_after, 10)
                print(f"  [429] rate limited — sleeping {wait}s")
                time.sleep(wait)
                continue
            if response.status_code != 200:
                raise ValueError(f"HTTP {response.status_code}: {_json.dumps(data)[:300]}")
            msg = data["choices"][0]["message"]
            content = msg.get("content") or msg.get("reasoning") or ""
            if not content.strip():
                raise ValueError("Empty response content")
            return content.strip()
        except Exception as e:
            last_error = e
            wait = 2 ** attempt
            print(f"  [Retry {attempt+1}/{max_retries}] {e} — waiting {wait}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed after {max_retries} retries. Last: {last_error}")


# Smoke test
print("Smoke test...")
test_response = classify_image_openrouter(test_split[0]["image"])
print(f"True: {id2label[test_split[0]['label']]}")
print(f"Raw response:\n{test_response}")
print("\nOpenRouter client ready.")

## 8. Ranking output parser (same logic as LoRA evaluation)

In [ ]:
CLASSES_LOWER = {l.lower(): l for l in new_labels}

def _extract_label(text: str):
    """Find the first matching cloud class label in text."""
    # Exact match
    for label in new_labels:
        if text.strip().lower() == label.lower():
            return label
    # Substring match (longest first to avoid partial clashes)
    for label in sorted(new_labels, key=len, reverse=True):
        if label.lower() in text.lower():
            return label
    return None

def get_top1_prediction(raw: str) -> str:
    """Extract predicted class from <answer> tags."""
    # Primary: inside <answer> tags
    tag_match = re.search(r"<answer>\s*(.+?)\s*</answer>", raw, re.IGNORECASE | re.DOTALL)
    if tag_match:
        label = _extract_label(tag_match.group(1))
        if label:
            return label

    # Fallback: scan full response
    label = _extract_label(raw)
    if label:
        return label

    return "UNPARSEABLE"


def get_full_ranking(raw: str) -> list:
    """Return a single-item list so downstream metrics degrade gracefully."""
    pred = get_top1_prediction(raw)
    return [pred] if pred != "UNPARSEABLE" else []


def format_rate(raw: str) -> bool:
    """True if the response contains a valid <answer> tag."""
    return bool(re.search(r"<answer>\s*(.+?)\s*</answer>", raw, re.IGNORECASE))


print("Classification parser ready (11-class direct match).")
print(f"Classes: {new_labels}")


## 9. Run inference on test set (collect full rankings)

In [ ]:
from tqdm.auto import tqdm
import json
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Configuration ─────────────────────────────────────────────────────────────
MAX_WORKERS = 10  # concurrent API calls; reduce if you see 429 rate-limit errors
RESUME      = False  # pick up from where we left off

# ── Load checkpoint ───────────────────────────────────────────────────────────
checkpoint_path = os.path.join(OUTPUT_DIR, "checkpoint_responses.jsonl")
done_indices = {}   # index → result dict

if RESUME and os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r") as f:
        for line in f:
            entry = json.loads(line)
            done_indices[entry["index"]] = entry
    print(f"Resuming: {len(done_indices)} already done out of {len(test_split)}")
elif os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print("Removed stale checkpoint, starting fresh.")

pending = [i for i in range(len(test_split)) if i not in done_indices]
print(f"Submitting {len(pending)} images with {MAX_WORKERS} workers…")

# ── Worker function ───────────────────────────────────────────────────────────
def process_one(i):
    example    = test_split[i]
    true_label = id2label[example["label"]]
    t0         = time.time()
    raw        = classify_image_openrouter(example["image"])
    elapsed    = time.time() - t0
    pred       = get_top1_prediction(raw)
    rank       = get_full_ranking(raw)
    return {
        "index":        i,
        "true_label":   true_label,
        "pred_label":   pred,
        "ranking":      rank,
        "raw_response": raw,
        "elapsed":      elapsed,
    }

# ── Parallel execution with live progress ────────────────────────────────────
_lock = threading.Lock()
checkpoint_file = open(checkpoint_path, "a")

try:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(process_one, i): i for i in pending}
        pbar = tqdm(total=len(test_split), initial=len(done_indices))
        for fut in as_completed(futures):
            try:
                result = fut.result()
            except Exception as exc:
                idx = futures[fut]
                print(f"  [ERROR idx={idx}] {exc}")
                pbar.update(1)
                continue

            with _lock:
                done_indices[result["index"]] = result
                correct = "Y" if result["true_label"] == result["pred_label"] else "N"
                pbar.set_postfix(last=f"{result['pred_label'][:15]} {correct}")
                pbar.update(1)
                checkpoint_file.write(json.dumps({
                    k: result[k] for k in ("index","true_label","pred_label","ranking","raw_response")
                }) + "\n")
                checkpoint_file.flush()
        pbar.close()
finally:
    checkpoint_file.close()

# ── Reassemble results in original order ─────────────────────────────────────
ordered = [done_indices[i] for i in range(len(test_split)) if i in done_indices]
true_labels_list = [r["true_label"]   for r in ordered]
pred_labels_list = [r["pred_label"]   for r in ordered]
rankings         = [r["ranking"]      for r in ordered]
raw_responses    = [r["raw_response"] for r in ordered]

accuracy_top1 = sum(t == p for t, p in zip(true_labels_list, pred_labels_list)) / len(true_labels_list)
fmt_rate      = sum(format_rate(r) for r in raw_responses) / len(raw_responses)
print(f"\nDone: {len(ordered)}/{len(test_split)} images")
print(f"Top-1 accuracy : {accuracy_top1:.4f}")
print(f"Format rate    : {fmt_rate:.4f}")

## 10. Classification report, confusion matrix & ranking metrics

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    balanced_accuracy_score, classification_report, cohen_kappa_score, confusion_matrix
)

# ── Top-k accuracies ──────────────────────────────────────────────────────────
top3  = sum(1 for t, r in zip(true_labels_list, rankings) if t in r[:3])  / len(true_labels_list)
top5  = sum(1 for t, r in zip(true_labels_list, rankings) if t in r[:5])  / len(true_labels_list)
top2  = sum(1 for t, r in zip(true_labels_list, rankings) if t in r[:2])  / len(true_labels_list)

# ── MRR ───────────────────────────────────────────────────────────────────────
mrr = np.mean([
    1.0 / (r.index(t) + 1) if t in r else 0.0
    for t, r in zip(true_labels_list, rankings)
])

# ── NDCG@11 ───────────────────────────────────────────────────────────────────
def ndcg_at_k(true_label, ranking, k=NUM_LABELS):
    if true_label not in ranking:
        return 0.0
    pos = ranking.index(true_label)
    if pos >= k:
        return 0.0
    return 1.0 / math.log2(pos + 2)   # ideal DCG = 1/log2(2) = 1.0

ndcg = np.mean([ndcg_at_k(t, r) for t, r in zip(true_labels_list, rankings)])

# ── Other metrics ─────────────────────────────────────────────────────────────
valid_mask   = [p in new_labels for p in pred_labels_list]
true_valid   = [t for t, v in zip(true_labels_list, valid_mask) if v]
pred_valid   = [p for p, v in zip(pred_labels_list, valid_mask) if v]
bal_acc = balanced_accuracy_score(true_valid, pred_valid)
kappa   = cohen_kappa_score(true_valid, pred_valid, labels=new_labels)
fmt_rate = sum(format_rate(r) for r in raw_responses) / len(raw_responses)

print("=" * 60)
print(f"  Top-1  : {accuracy_top1:.4f}   Top-2  : {top2:.4f}")
print(f"  Top-3  : {top3:.4f}   Top-5  : {top5:.4f}")
print(f"  MRR    : {mrr:.4f}   NDCG@11: {ndcg:.4f}")
print(f"  Bal Acc: {bal_acc:.4f}   Cohen κ: {kappa:.4f}")
print(f"  Format : {fmt_rate:.4f}")
print("=" * 60)

# ── Classification report ─────────────────────────────────────────────────────
report = classification_report(true_valid, pred_valid, labels=new_labels, digits=3, zero_division=0)
print(f"\nClassification Report\n{report}")

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(true_valid, pred_valid, labels=new_labels)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=new_labels, yticklabels=new_labels,
            linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted (Top-1)"); ax.set_ylabel("True")
ax.set_title(
    f"Confusion Matrix | Qwen3.5-27B Zero-Shot (11 Cloud Classes) | "
    f"Top-1={accuracy_top1:.3f} | MRR={mrr:.3f} | Bal Acc={bal_acc:.3f}"
)
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0); plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{TAG}.png")
plt.savefig(cm_path, dpi=150); plt.show()
print(f"Saved: {cm_path}")


## 11. Per-class accuracy, mean rank & nDCG@11

In [ ]:
# ── Per-class metrics table ───────────────────────────────────────────────────
per_class_top1      = {}
per_class_mean_rank = {}
per_class_ndcg      = {}

for label in new_labels:
    idxs = [i for i, t in enumerate(true_labels_list) if t == label]
    if not idxs:
        per_class_top1[label] = float("nan")
        per_class_mean_rank[label] = float("nan")
        per_class_ndcg[label] = float("nan")
        continue
    per_class_top1[label]      = sum(1 for i in idxs if pred_labels_list[i] == label) / len(idxs)
    per_class_mean_rank[label] = np.mean(
        [rankings[i].index(label) + 1 if label in rankings[i] else NUM_LABELS + 1 for i in idxs]
    )
    per_class_ndcg[label]      = np.mean([ndcg_at_k(label, rankings[i]) for i in idxs])

print(f"  {'Class':30s}  {'Top-1':>7s}  {'MeanRank':>9s}  {'nDCG@11':>9s}")
print("─" * 62)
for label in new_labels:
    t1 = per_class_top1[label]
    mr = per_class_mean_rank[label]
    nd = per_class_ndcg[label]
    print(f"  {label:30s}  {t1:>7.3f}  {mr:>9.2f}  {nd:>9.4f}")
print("─" * 62)
print(
    f"  {'MACRO AVG':30s}  {accuracy_top1:>7.3f}  "
    f"{np.nanmean(list(per_class_mean_rank.values())):>9.2f}  "
    f"{np.nanmean(list(per_class_ndcg.values())):>9.4f}"
)

# ── Per-class Top-1 bar chart ─────────────────────────────────────────────────
labels_sorted = sorted(new_labels, key=lambda l: per_class_top1.get(l, 0), reverse=True)
vals_sorted   = [per_class_top1.get(l, 0) for l in labels_sorted]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels_sorted, vals_sorted, color="#2196F3", edgecolor="black", linewidth=0.5)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Top-1 Accuracy")
ax.set_title(
    f"Per-Class Top-1 Accuracy — Qwen3.5-27B Zero-Shot | 11 Cloud Classes ({len(true_labels_list)} images)"
)
plt.xticks(rotation=45, ha="right")
for bar, val in zip(bars, vals_sorted):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
        f"{val:.2f}", ha="center", va="bottom", fontsize=8
    )
plt.tight_layout()
acc_path = os.path.join(OUTPUT_DIR, f"per_class_accuracy_{TAG}.png")
plt.savefig(acc_path, dpi=150); plt.show()
print(f"Saved: {acc_path}")


## 12. Save results, metrics & final summary

In [ ]:
import json as _json

# ── Save predictions CSV ──────────────────────────────────────────────────────
positions = [r.index(t) + 1 if t in r else NUM_LABELS + 1 for t, r in zip(true_labels_list, rankings)]

results_df = pd.DataFrame({
    "true_label":   true_labels_list,
    "pred_label":   pred_labels_list,
    "correct":      [t == p for t, p in zip(true_labels_list, pred_labels_list)],
    "true_rank":    positions,
    "ranking":      ["|".join(r) for r in rankings],
    "raw_response": raw_responses,
})
csv_path = os.path.join(OUTPUT_DIR, f"predictions_{TAG}.csv")
results_df.to_csv(csv_path, index=False)
print(f"Saved predictions: {csv_path}")

# ── Save metrics JSON ─────────────────────────────────────────────────────────
metrics = {
    "model":        MODEL,
    "n_images":     len(true_labels_list),
    "n_classes":    NUM_LABELS,
    "taxonomy":     "11 standard cloud classes (Altocumulus, Altostratus, Cirrocumulus, Cirrus, Clear Sky, Contrails, Cumulonimbus, Cumulus, Stratocumulus, Stratus, Veil)",
    "classes":      new_labels,
    "top1":         round(accuracy_top1, 6),
    "top2":         round(top2, 6),
    "top3":         round(top3, 6),
    "top5":         round(top5, 6),
    "mrr":          round(float(mrr), 6),
    "ndcg":         round(float(ndcg), 6),
    "balanced_acc": round(float(bal_acc), 6),
    "cohen_kappa":  round(float(kappa), 6),
    "format_rate":  round(fmt_rate, 6),
}
metrics_path = os.path.join(OUTPUT_DIR, f"metrics_{TAG}.json")
with open(metrics_path, "w") as f:
    _json.dump(metrics, f, indent=2)
print(f"Saved metrics:     {metrics_path}")

# ── Final summary ─────────────────────────────────────────────────────────────
print()
print("=" * 65)
print("  FINAL RESULTS — Qwen3.5-27B Zero-Shot (11 Standard Cloud Classes)")
print("=" * 65)
print(f"  Classes: {new_labels}")
print(f"  Images : {len(true_labels_list)}")
print()
print(f"  Top-1: {accuracy_top1:.4f} | Top-2: {top2:.4f} | Top-3: {top3:.4f} | Top-5: {top5:.4f}")
print(f"  MRR: {mrr:.4f} | NDCG@11: {ndcg:.4f} | Bal Acc: {bal_acc:.4f} | Cohen κ: {kappa:.4f}")
print(f"  Format rate: {fmt_rate:.4f}")
print("=" * 65)
